# Reflecting on Qwen3.5 answers using SMT

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

This notebook demonstrates how to exploit mathematically verified contradictory cases detected by the SMT solver to generate more reliable answers.

In [ ]:
import json
import random
import re
from pathlib import Path

from IPython.display import display
from PIL import Image

from staged_qwen3_5_scivqa.config import (
    COMPETITION_DATA_DIR,
    DATA_DIR,
    PROMPT_REWRITE,
    ENABLE_THINKING,
    REFLECTION_MAX_NEW_TOKENS,
    REFLECTION_MAX_SEQUENCE_LENGTH,
    REFLECTION_TEMPERATURE,
    REFLECTION_TOP_P,
    REFLECTION_TOP_K,
    REFLECTION_MIN_P,
    REFLECTION_REPETITION_PENALTY,
)
from staged_qwen3_5_scivqa.models.reflection_runner import load_reflection_model
from staged_qwen3_5_scivqa.analysis import find_contradictory_answers

In [ ]:
MODEL_ID = "unsloth/Qwen3.5-9B"
DATA_DIR.mkdir(exist_ok=True)
CATEGORY = "test"

SPLIT_DIR = COMPETITION_DATA_DIR / CATEGORY

INITIAL_STATE_PATH = DATA_DIR / f"submission_finetuning_{CATEGORY}_state.json"
SMT_FILE = DATA_DIR / f"smt_{CATEGORY}_state.json"

In [ ]:
with open(SMT_FILE) as f:
    smt_data = json.load(f)

with open(INITIAL_STATE_PATH) as f:
    initial_state = json.load(f)

In [ ]:
candidates = find_contradictory_answers(
    initial_state, smt_data, answer_types=["Yes/No"]
)

if not candidates:
    raise ValueError(
        "No contradictory Yes/No answers found in the dataset."
    )

selected_candidate = random.choice(candidates)
target_sample_id = selected_candidate["sample_id"]
target_sub_fig = selected_candidate["sub_fig"]
target_q_obj = selected_candidate["q_obj"]
smt_entry = selected_candidate["smt_entry"]

print(f"Sample ID: {target_sample_id}")
print(f"Subfigure: {target_sub_fig}")
print(f"Question: {target_q_obj['question']}")
print(f"Initial Answer: {target_q_obj['answer']}")
print("-" * 60)

In [ ]:
target_json_path = None
target_data = None

for json_file in SPLIT_DIR.rglob("*.json"):
    if (
        "content.json" in json_file.name
        or "images" not in str(json_file)
        or ".vscode" in str(json_file)
    ):
        continue

    with open(json_file) as f:
        data = json.load(f)

    if data.get("sample_id") == target_sample_id:
        target_json_path = json_file
        target_data = data
        break

img_path = target_json_path.with_suffix(".jpg")
full_img = Image.open(img_path)
box = target_data["bbox"][target_sub_fig]
crop = full_img.crop((box["x"], box["y"], box["x"] + box["width"], box["y"] + box["height"]))
display(crop)

print("-" * 60)
print(smt_entry["code"][smt_entry["code"].find(";; --- [PASS 2: Logic & Execution]") :])
print(f">> {smt_entry['output'].split(chr(10)).pop(1)}")

In [ ]:
rewrite_prompt = PROMPT_REWRITE.format(
    question_type=target_q_obj.get("question_type", ""),
    question=target_q_obj.get("question", ""),
    answer_type=target_q_obj.get("answer_type", ""),
    answer_cache=target_q_obj.get("answer", ""),
    code=smt_entry.get("code", "N/A"),
    output=smt_entry.get("output", "N/A"),
)

print("--- CONSTRUCTED PROMPT ---")
print(rewrite_prompt)

In [ ]:
model, tokenizer = load_reflection_model(
    MODEL_ID, max_seq_length=REFLECTION_MAX_SEQUENCE_LENGTH
)

In [ ]:
messages = [{"role": "user", "content": rewrite_prompt}]

input_text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)

inputs = tokenizer(text=input_text, return_tensors="pt").to("cuda")

output_ids = model.generate(
    **inputs,
    max_new_tokens=REFLECTION_MAX_NEW_TOKENS,
    use_cache=True,
    temperature=REFLECTION_TEMPERATURE,
    min_p=REFLECTION_MIN_P,
    top_p=REFLECTION_TOP_P,
    top_k=REFLECTION_TOP_K,
    repetition_penalty=REFLECTION_REPETITION_PENALTY,
)

full_response = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
)

In [ ]:
initial = target_q_obj["answer"]

print("--- FULL MODEL RESPONSE ---")
print(full_response)
print("--- INITIAL ANSWER ---")
print(initial)

In [ ]:
match = re.search(r"<ANSWER>(.*?)</ANSWER>", full_response, re.DOTALL | re.IGNORECASE)

parsed_answer, hit_max_tokens = "", False
if match:
    parsed_answer = match.group(1).strip()
    hit_max_tokens = False

is_empty = len(parsed_answer) == 0
is_rambling = len(parsed_answer) > (2 * len(initial))
is_too_short = target_q_obj["answer_type"] not in {"Factoid", "Yes/No"} and len(
    parsed_answer
) < (0.4 * len(initial))

if is_empty or hit_max_tokens or is_rambling or is_too_short:
    reason = (
        "Empty"
        if is_empty
        else (
            "Hit Max/No Marker"
            if hit_max_tokens
            else ("Rambling" if is_rambling else "Too Short")
        )
    )
    print(f"FALLBACK TRIGGERED. Reason: {reason}")
    final_answer = initial
else:
    print("SUCCESS. Answer reflected.")
    final_answer = parsed_answer

print(f"\n--- FINAL ANSWER ---\n{final_answer}")